# Richter 2020 variant dataset

Builds the annotated de novo variant (DNV) tables of CHD probands and controls
from Richter et al. 2020 that are used as VEP input and for all Richter analyses:

1. Load the case and control DNVs (Supplementary Tables S2, S3) and the CHD gene lists (S5).
2. Exclude probands with known or likely causal coding variants and controls with a CHD history.
3. Annotate ENCODE4 cCREs, GTF features, phyloP, coding effects, CADD, overlap with cardioid
   ATAC peaks and ABC candidate enhancers, and the SDR007 selection.
4. Align variants reported on the opposite strand to the reference genome.
5. Build the CRISPRi screen VCF (CHD SNVs in cardioid enhancers and all SNVs within ±5 bp).

**Outputs** (in `variant_data_dir`; the writing lines are commented out):
`VCF_all_Richter_vars.csv`, `VCF_new_enhancer_Richter_vars.csv`,
`VCF_new_enhancer_Richter_vars_for_CRISPRi.csv`.

**Paths** come from `config/paths.yaml` (`variant_data_dir`, `atac_peak_dir`,
`abc_enhancer_table`, `genome_fasta`, `gtf`). CADD scores for indels are computed on the
CADD website (see below).

In [ ]:
import glob
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyBigWig
import seaborn as sns
from pyfaidx import Fasta

from cardiomics_s2f.config import load_paths
from cardiomics_s2f.eval.track_prediction import get_gtf
from cardiomics_s2f.io.variant_annotation import (
    add_variant_id,
    align_variants_to_reference_strand,
    annotate_cadd_snv_scores,
    annotate_cardioid_enhancer_overlap,
    annotate_cCREs,
    annotate_coding_effect,
    annotate_gtf_features,
    annotate_phyloP_score,
    celltype_from_peak_file,
    flag_variants_in_peaks,
    load_ccre_data,
    read_narrowpeak,
    vcf_to_pr,
)

# Patient data overview
taken from Richter et al 2020:
Coding de novo variants identified by whole-exome sequencing and whole-genome sequencing
WES data were available for 612 of 763 CHD probands4,5. Among 628 coding DNVs, including 582 within WES capture regions (lifted over14 to hg38), both WES and WGS identified 509 (81%), whereas 38 of 69 DNVs called only by WES were confirmed by WGS IGV visualization (Supplementary Note). Fifty coding DNVs identified solely by WGS (8%; 0.08/proband) included 4 within and 46 outside WES capture regions. One DNV that was initially called by WES was removed for low read depth; three were not called by WES but were confirmed by WES IGV visualization.

These analyses defined damaging DNVs in established CHD genes (PTPN11, NOTCH1 (n = 2), FBN1, FLT4, NR2F2 and GATA4), and identified six individuals with 22q11 copy number variants and one with trisomy 21. The proband with a previously reported pathogenic FBN1 DNV in exon 42 (1-00761) had mitral stenosis, brachycephaly, short stature and other features consistent with geleophysic dysplasia (MIM 614185), 50% of cases of which are caused by damaging DNVs in FBN1 exon 41 or 42. Damaging DNVs in known CHD genes were confirmed with reference-free DNV calling (Methods) and IGV visualization. Six potentially damaging DNVs were identified in candidate CHD genes, including one insertion that was detected only with reference-free calling (Supplementary Table 4), but these individuals were retained for noncoding analyses. Following the exclusion of probands with probable causal genetics, 749 CHD probands were analyzed for noncoding DNVs.

Quantitative burden with categorical de novo variant classifications
We observed no noncoding DNV enrichment in 749 CHD trios for DNVs that are associated with human (n = 210) or mouse (n = 614) CHD genes or genes that are highly expressed in heart development (n = 4,420) (Supplementary Tables 5 and 6). Similarly, we observed no enrichment in noncoding cardiac regulatory features comprising transcription factor binding sites (nhuman = 8, nmouse = 45), histone marks (nhuman = 45, nmouse = 60) and DNase hypersensitivity sites (nhuman = 23, nmouse = 3) assayed on cardiac cells (nhuman = 15), prenatal or fetal heart tissue (nhuman = 26, nmouse = 34) and postnatal heart tissue (nhuman = 35, nmouse = 74) (Methods; Extended Data Fig. 3 and Supplementary Table

In [ ]:
total_CHD_probands = 763
n_noncausal_CHD_probands = 749
n_causal_CHD_probands = total_CHD_probands - n_noncausal_CHD_probands

n_CHD_DNVs = 56106
n_ctrl_DNVs = 114065
n_non_coding_DNVs_implicated_with_CHD_genes = pd.DataFrame({
    "Gene List": [ 
        "Chromatin Modification", 
        "Cilia", 
        "Highly Expressed in the Developing Heart", 
        "OMIM Human CHD", 
        "Known Mouse CHD", 
        "Genes with Multiple Damaging DNVs (Jin 2017)",
        "Non implicated"
    ],
    "CHD DNVs": [215, 123, 1825, 88, 260, 43, n_CHD_DNVs - 2554],
    "Control DNVs": [536, 291, 3569, 187, 504, 64, n_ctrl_DNVs - 5151]
})
n_non_coding_DNVs_implicated_with_CHD_genes

# Loading variant data

In [ ]:
paths = load_paths()
VARIANT_DIR = paths.variant_data_dir  # Richter tables, annotation tracks and the VCFs written below
PEAK_DIR = paths.atac_peak_dir        # pseudobulk ATAC narrowPeak files

In [ ]:
#patient ids of patiants that have DNVs or other mutation that are in codign regions or othr likely causal CHD genes
coding_DNV_patient_ids =  ["1-09585","1-03825","1-10377","1-06313","1-05007","1-02812","1-05460","1-00761","1-10183","1-13136","1-01384","1-07749","1-13185"]

In [ ]:
Richter_CHD_vars = pd.read_excel(
    VARIANT_DIR / 'Richter2020_Supp_Tables.xlsx',
    sheet_name='S2 DNVs in Cases',
    usecols="A:G",      # Limits the extraction to columns A through G
    skiprows=1,         # Skips row 1, making row 2 your header
    nrows=58091        
)
Richter_control_vars = pd.read_excel(
    VARIANT_DIR / 'Richter2020_Supp_Tables.xlsx',
    sheet_name='S3 DNVs in Controls',
    usecols="A:G",      # Limits the extraction to columns A through G
    skiprows=1,        
    nrows=117345        
)

In [ ]:
CHD_genes = pd.read_excel(
    VARIANT_DIR / 'Richter2020_Supp_Tables.xlsx',
    sheet_name='S5 CHD gene lists',
    usecols="A:F",     
    skiprows=1,         # Skips row 1, making row 2 your header
    nrows=4420
)

In [ ]:
CHD_genes

In [ ]:
#drop all varints from patiants that have coding mutations or are trissomy or duplications
Richter_CHD_vars = Richter_CHD_vars[(Richter_CHD_vars["Known cause of CHD (cases)"].isna()) |~(Richter_CHD_vars["Blinded ID"].isin(coding_DNV_patient_ids))].drop(
    ["Known cause of CHD (cases)", "Past medical history of CHD (controls)"], axis = 1)

In [ ]:
#drop control with past CHD history
Richter_control_vars = Richter_control_vars[Richter_control_vars["Past medical history of CHD (controls)"].isna()].drop(
    ["Past medical history of CHD (controls)"], axis = 1)    

## Add annotations

In [ ]:
ctrl_pr = vcf_to_pr(Richter_control_vars)
chd_pr = vcf_to_pr(Richter_CHD_vars)

In [ ]:
ccre_pr = load_ccre_data(VARIANT_DIR / "hg38_ENCODE4_cCREs.gz")
Richter_control_vars = annotate_cCREs(ccre_pr, ctrl_pr)
Richter_CHD_vars = annotate_cCREs(ccre_pr, chd_pr)

In [ ]:
## Add GTF fiel information for exons introns utrs etc.
GTF_FILE = paths.gtf
gtf = get_gtf(GTF_FILE, chrom_filter="autosomesXY", gene_type_filter=None, feature=None)

In [ ]:
Richter_CHD_vars[['gtf_feature', 'gene_name', 'gene_type', 'closest_TSS']] = annotate_gtf_features(
    Richter_CHD_vars, gtf
)

Richter_control_vars[['gtf_feature', 'gene_name', 'gene_type', 'closest_TSS']] = annotate_gtf_features(
    Richter_control_vars, gtf
)

In [ ]:
bw_cons = pyBigWig.open(str(VARIANT_DIR / "hg38.phyloP100way.bw"))

Richter_CHD_vars['phyloP_100way'] = annotate_phyloP_score(Richter_CHD_vars, bw_cons)
Richter_control_vars['phyloP_100way'] = annotate_phyloP_score(Richter_control_vars, bw_cons)

bw_cons.close()

In [ ]:
Richter_CHD_vars[['all_effects', 'coding_effect', 'aa_change', 'impact']]  = annotate_coding_effect(Richter_CHD_vars)

In [ ]:
Richter_control_vars[['all_effects', 'coding_effect', 'aa_change', 'impact']]  = annotate_coding_effect(Richter_control_vars)

In [ ]:
# Richter_CHD_vars[(Richter_CHD_vars.gene_name.isin(CHD_genes["Human CHD Genes (210)"].dropna())) &
#                 (Richter_CHD_vars.gtf_feature.isin(["CDS", "exon", "start_codon", "stop_codon"]))]

Richter_CHD_vars.loc[
    (Richter_CHD_vars.gene_name.isin(CHD_genes["Human CHD Genes (210)"].dropna())) &
    (Richter_CHD_vars.gtf_feature.isin(["CDS", "exon", "start_codon", "stop_codon"])) &
    (Richter_CHD_vars.impact == "HIGH"),
    ["cCRE_class", "gtf_feature","gene_name", "gene_type","phyloP_100way", "coding_effect", "aa_change", "all_effects", "impact"]
]

In [ ]:
print(Richter_CHD_vars.loc[
    (Richter_CHD_vars.gene_name.isin(CHD_genes["Human CHD Genes (210)"].dropna())) &
    (Richter_CHD_vars.gtf_feature.isin(["CDS", "exon", "start_codon", "stop_codon"])) &
    (Richter_CHD_vars.impact == "HIGH"),
    ["all_effects"]
].values)

In [ ]:
Richter_control_vars.loc[
    (Richter_control_vars.gene_name.isin(CHD_genes["Human CHD Genes (210)"].dropna())) &
    (Richter_control_vars.gtf_feature.isin(["CDS", "exon", "start_codon", "stop_codon"])) &
    (Richter_control_vars.impact == "HIGH"),
    ["cCRE_class", "gtf_feature","gene_name", "gene_type","phyloP_100way", "coding_effect", "aa_change", "all_effects", "impact"]
]

In [ ]:
print(Richter_control_vars.loc[
    (Richter_control_vars.gene_name.isin(CHD_genes["Human CHD Genes (210)"].dropna())) &
    (Richter_control_vars.gtf_feature.isin(["CDS", "exon", "start_codon", "stop_codon"])) &
    (Richter_control_vars.impact == "HIGH"),
    ["all_effects"]
].values)

**Based on these annoations I exclude the patients carrying the three muations shown above in the CHD cohort as htery are very cocnerved and linked to CHD before so Id argue they are causal. As for the control variant its liky less causal becuase of the negative Phylo score and the distal enhancer might suggest that ths region as become more regualtory than coding over time.**

In [ ]:
#excludign patients with causal vars
casual_patients = list(Richter_CHD_vars.loc[
    (Richter_CHD_vars.gene_name.isin(CHD_genes["Human CHD Genes (210)"].dropna())) &
    (Richter_CHD_vars.gtf_feature.isin(["CDS", "exon", "start_codon", "stop_codon"])) &
    (Richter_CHD_vars.impact == "HIGH"),
    ["Blinded ID"]
].values.flatten())

Richter_CHD_vars = Richter_CHD_vars[~Richter_CHD_vars["Blinded ID"].isin(casual_patients)]

# Add Intersection with chromatin acceible regions from scMultiome ATAC data

In [ ]:
Richter_CHD_vars = add_variant_id(Richter_CHD_vars)
Richter_control_vars = add_variant_id(Richter_control_vars)

In [ ]:
#make vcf file for all richter variants
Richter_CHD_vars["Cohort"]  = "CHD"
Richter_control_vars["Cohort"] = "Control"
Richter_all_vcf = pd.concat([Richter_CHD_vars, Richter_control_vars], ignore_index=True)

Richter_all_vcf["Start"] = Richter_all_vcf["Pos"] - 1
Richter_all_vcf["End"] = Richter_all_vcf["Pos"]

Richter_all_vcf = add_variant_id(Richter_all_vcf)

In [ ]:
#add SDR007 annot
CHD_vars_selected = pd.read_csv(VARIANT_DIR / "SDR007_Varaints_sel_NGG_NG.csv") #list of vars selcted in SDR007
CHD_vars_selected = CHD_vars_selected.rename(
    columns={"CHROM": "Chromosome", "REF": "Ref", "ALT": "Alt"}
)
CHD_vars_selected["Pos"] = CHD_vars_selected["end"].astype(int)
CHD_vars_selected = add_variant_id(CHD_vars_selected)

Richter_all_vcf["in_SDR007"] = Richter_all_vcf.variant_id.isin(CHD_vars_selected["variant_id"])
Richter_all_vcf["is_CHD"] = Richter_all_vcf.Cohort == "CHD"

In [ ]:
#get ATAC peaks
peak_files = glob.glob(os.path.join(PEAK_DIR, "*_peaks.narrowPeak"))
cell_types = {celltype_from_peak_file(p): p for p in peak_files}
print("Cell types found:", list(cell_types.keys()))

CA_variants = {}
for ct, pf in cell_types.items():
    peaks = read_narrowpeak(pf)
    mask  = flag_variants_in_peaks(Richter_all_vcf, peaks)
    ct_results = {}
    for cohort, grp in Richter_all_vcf.groupby("Cohort", observed=False):
        grp_mask = mask[grp.index]
        ct_results[cohort] = {
            "in":  grp[grp_mask],
            "out": grp[~grp_mask],
        }
    CA_variants[ct] = ct_results
    print(f"  {ct}: done")

In [ ]:
#get teh vairnts that show up in one CA region in at least one cell type
CHD_CA_var_ids = set.union(*[set(res['CHD']["in"]["variant_id"]) for res in CA_variants.values()])
ctrl_CA_var_ids = set.union(*[set(res['Control']["in"]["variant_id"]) for res in CA_variants.values()])
    
print(len(CHD_CA_var_ids), len(ctrl_CA_var_ids))

In [ ]:
Richter_CHD_vars["is_cardioid_CA"] = Richter_CHD_vars.variant_id.isin(CHD_CA_var_ids)
Richter_control_vars["is_cardioid_CA"] = Richter_control_vars.variant_id.isin(ctrl_CA_var_ids)
Richter_all_vcf["is_cardioid_CA"] = Richter_all_vcf.variant_id.isin(ctrl_CA_var_ids | CHD_CA_var_ids)

## Add cardioid enhacer data

In [ ]:
#enhacer region from abc model
enhancer_regions = pd.read_csv(paths.abc_enhancer_table, sep = "\t")

#collapse indiviudal days as this is still on the day split annotation
cts = []
for s in enhancer_regions.samples:
    d_cts = s.split(",")
    ct = {d_ct.strip().split('_', 1)[1] for d_ct in d_cts}
    cts.append(ct)
enhancer_regions["celltype"] = cts 

enhancer_vars = annotate_cardioid_enhancer_overlap(variants_df = Richter_all_vcf, regions_df = enhancer_regions)
enhancer_vars["in_cardioid_enhancer"] = True

In [ ]:
Richter_all_vcf = pd.merge(Richter_all_vcf,
         enhancer_vars[["Chromosome", "Pos", "Ref","Alt", "variant_id", "enhancer_regulatory_position","is_cardioid_H3K27ac","celltypes_in_enhancer", "in_cardioid_enhancer"]], 
                        on=["Chromosome", "Pos", "Ref","Alt", "variant_id"], 
                        how='left')

Richter_all_vcf = Richter_all_vcf.fillna({"in_cardioid_enhancer": False})

In [ ]:
Richter_CHD_vars = pd.merge(Richter_CHD_vars,
         enhancer_vars[["Chromosome", "Pos", "Ref","Alt", "variant_id", "enhancer_regulatory_position","is_cardioid_H3K27ac","celltypes_in_enhancer", "in_cardioid_enhancer"]], 
                        on=["Chromosome", "Pos", "Ref","Alt", "variant_id"], 
                        how='left')

Richter_CHD_vars = Richter_CHD_vars.fillna({"in_cardioid_enhancer": False})
Richter_control_vars = pd.merge(Richter_control_vars,
         enhancer_vars[["Chromosome", "Pos", "Ref","Alt", "variant_id", "enhancer_regulatory_position","is_cardioid_H3K27ac","celltypes_in_enhancer", "in_cardioid_enhancer"]], 
                        on=["Chromosome", "Pos", "Ref","Alt", "variant_id"], 
                        how='left')

Richter_control_vars = Richter_control_vars.fillna({"in_cardioid_enhancer": False})

In [ ]:
#filter based on refernce genome and resolve reverse complementary mismatched
hg38_path = str(paths.genome_fasta)
hg38 = Fasta(hg38_path)

Richter_all_vcf = align_variants_to_reference_strand(Richter_all_vcf, hg38 = hg38)
Richter_all_vcf = Richter_all_vcf.drop_duplicates()

In [ ]:
#make vcf file for all enhacer richter variants
Richter_enhancer_vcf = Richter_all_vcf[Richter_all_vcf.in_cardioid_enhancer]
Richter_enhancer_vcf.head()

## CADD computation
I also want to include CADD scores for all variants this is best done through htere website for which we need a gzip fiel of max 100_000 vars

In [ ]:
CADD_indels = Richter_all_vcf.loc[~(Richter_all_vcf.Ref.str.len() == 1) | ~(Richter_all_vcf.Alt.str.len() == 1),
                           ['Chromosome', 'Pos', 'variant_id', 'Ref', 'Alt']
                          ]

chunks = np.array_split(CADD_indels, len(CADD_indels) // 1000)
for i, chunk in enumerate(chunks):
    filename = VARIANT_DIR / f"indels_for_cadd_chunk{i}.tsv.gz"
    # chunk.to_csv(filename, sep='\t', index=False, header=False, compression='gzip')
    # print(f"Exported {len(chunk)} variants to {filename}")

Upload to https://cadd.gs.washington.edu/score and run it. Quick note If i do this in the future: This took hours so probabaly looking up the SNVs from UCSC table browser and only submitting the indels and MNVs would be a lot faster.

In [ ]:
#read the indel prediction once their done
CADD_indels = []
for f in (VARIANT_DIR / "CADD_indels_res").glob("*.tsv.gz"):
    indels = pd.read_csv(f, sep='\t', comment='#', compression=None, names=['Chromosome', 'Pos', 'Ref', 'Alt', 'RawScore', 'CADD_PHRED_score']).drop('RawScore', axis = 1)
    CADD_indels.append(indels)

CADD_indels = pd.concat(CADD_indels)
CADD_indels["Chromosome"] = ["chr" + str(c) for c in CADD_indels["Chromosome"]]
CADD_indels.head(5)

In [ ]:
Richter_all_vcf = pd.merge(Richter_all_vcf, CADD_indels, 
                        on=['Chromosome', 'Pos', 'Ref', 'Alt'], 
                        how='left')

In [ ]:
#Using precomputed SNVs
CADD_paths = {
    'A': str(VARIANT_DIR / 'CADD_snv/a.bw'),
    'C': str(VARIANT_DIR / 'CADD_snv/c.bw'),
    'G': str(VARIANT_DIR / 'CADD_snv/g.bw'),
    'T': str(VARIANT_DIR / 'CADD_snv/t.bw'),
}

df_cadd_scored = annotate_cadd_snv_scores(Richter_all_vcf, CADD_paths)

In [ ]:
cadd_merge = pd.merge(Richter_all_vcf, df_cadd_scored[['Chromosome', 'Pos', 'Ref', 'Alt', 'CADD_PHRED_score']], 
                        on=['Chromosome', 'Pos', 'Ref', 'Alt'], 
                        how='left')

In [ ]:
cadd_merge['CADD_PHRED_score'] = cadd_merge['CADD_PHRED_score_x'].fillna(cadd_merge['CADD_PHRED_score_y'])

In [ ]:
Richter_all_vcf = cadd_merge.drop(['CADD_PHRED_score_x','CADD_PHRED_score_y'], axis = 1)

In [ ]:
sns.histplot(data=Richter_all_vcf, x='CADD_PHRED_score', hue='Cohort', element='step', stat='density',
                     common_norm=False,
                     log_scale=False, bins=50)
plt.show()

In [ ]:
Richter_all_vcf = Richter_all_vcf.rename(columns={'Chromosome': "CHROM", 'Pos': "POS", 'Ref': "REF", 'Alt': "ALT"})
Richter_enhancer_vcf = Richter_enhancer_vcf.rename(columns={'Chromosome': "CHROM", 'Pos': "POS", 'Ref': "REF", 'Alt': "ALT"})

In [ ]:
#save the new varaints list with metadata to disk
#Richter_CHD_vars.to_csv(VARIANT_DIR / "Richter2020_CHD_vars_complete_list_metadata.csv", sep = ",", index=False)
#Richter_control_vars.to_csv(VARIANT_DIR / "Richter2020_CHD_controls_complete_list_metadata.csv", sep = ",", index=False)
#Richter_all_vcf.to_csv(VARIANT_DIR / "VCF_all_Richter_vars.csv", header=True, index = False)
#Richter_enhancer_vcf.to_csv(VARIANT_DIR / "VCF_new_enhancer_Richter_vars.csv", header=True, index = False)

## VCF for the CRISPRi screen design

CHD-only SNVs in cardioid enhancers, plus every possible SNV within ±5 bp of each
(a mini ISM window), to estimate how much of the local effect each Richter variant explains.

In [ ]:
#make list of variants for the varaint slection of the CRSIPRi screening
#for those pick the enhancer var positoves and extend in a +/- 5bp window and predict every muation and then report the percentage of effect expained

n = 5 #set to most common motif length - 1

Richter_enhancer_vcf = pd.read_csv(VARIANT_DIR / "VCF_new_enhancer_Richter_vars.csv")
CRISPRi_vars = Richter_enhancer_vcf[Richter_enhancer_vcf.is_CHD] #remove negative control vars
CRISPRi_vars = CRISPRi_vars[(CRISPRi_vars.REF.str.len() == 1) & (CRISPRi_vars.ALT.str.len() == 1)] #remove indels

new_pos = []
new_ref = []
new_var_id = []
new_chrom = []
new_sdr = []

#make adjecnet vars to estimate a mini ISM in the window of +- n aournd each richter var
for __, row in CRISPRi_vars.iterrows():
    chrom = row["CHROM"]
    pos = row["POS"]
    var_id = f"{row['variant_id']}_adjacent" 
    
    upstream_nt = str(hg38[chrom][pos-n-1:pos-1]).upper()
    downstream_nt = str(hg38[chrom][pos:pos+n]).upper()
    
    upstream_pos = list(range(pos-n, pos, 1))
    downstream_pos = list(range(pos+1, pos+n+1, 1))
    
    new_pos.extend(upstream_pos + downstream_pos)
    new_ref.extend(list(upstream_nt + downstream_nt)) 
    new_var_id.extend([var_id] * 2 * n)
    new_chrom.extend([chrom] * 2 * n)
    new_sdr.extend([row["in_SDR007"]] * 2 * n)
    
CRISPRi_vars_adj = pd.DataFrame({"variant_id": new_var_id, "CHROM": new_chrom, "POS": new_pos, "REF": new_ref, "in_SDR007": new_sdr})

alt_bases = {'A': ['C', 'G', 'T'],'C': ['A', 'G', 'T'],'G': ['A', 'C', 'T'], 'T': ['A', 'C', 'G']}

CRISPRi_vars_adj['ALT'] = CRISPRi_vars_adj['REF'].map(alt_bases)
CRISPRi_vars_adj = CRISPRi_vars_adj.explode('ALT', ignore_index=True)

#define variant ids
CRISPRi_vars_adj["ref_variant_id"] = CRISPRi_vars_adj["variant_id"].str.split("_").str[0]
CRISPRi_vars_adj["variant_id"] = CRISPRi_vars_adj["CHROM"] + "-" + CRISPRi_vars_adj["POS"].astype(str) + "-" + CRISPRi_vars_adj["REF"] + "-" + CRISPRi_vars_adj["ALT"] + "_" + CRISPRi_vars_adj["variant_id"] 
CRISPRi_vars["ref_variant_id"] = CRISPRi_vars["variant_id"]

#merge with richter vars
CRISPRi_vcf = pd.concat([CRISPRi_vars, CRISPRi_vars_adj]).drop(["is_CHD", "REF_hg38", "REF_rev_comp"], axis = 1).sort_values(by = ["CHROM", "POS"],ignore_index=True)

CRISPRi_vcf

In [ ]:
# CRISPRi_vcf.to_csv(VARIANT_DIR / "VCF_new_enhancer_Richter_vars_for_CRISPRi.csv", header=True, index = False)